# Red Five — signal report

Synthetic, descriptive evidence only. No acceptance verdict, inferred weights, residualizer fitting, NAV path or significance claims. Run with the project's Jupyter environment. Replace the generated path with your own verified report to explore real data; keep private outputs out of Git.

In [ ]:
%matplotlib inline
from pathlib import Path

from IPython.display import display

from red_five.evaluation import evaluate
from red_five.reporting import publish, seal_report, software_versions, source_identity
from red_five.visualization import load_report

ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "examples/evaluation.json").is_file()
)

## Generate a synthetic example

This calls production functions; the notebook does not recalculate metrics. Versioned output paths keep reruns idempotent without overwriting different evidence.

In [ ]:
result = evaluate(
    (ROOT / "examples/signals.csv").read_bytes(),
    (ROOT / "examples/evaluation.json").read_bytes(),
    (ROOT / "STATISTICAL_ANALYSIS_PLAN.md").read_bytes(),
    (ROOT / "uv.lock").read_bytes(),
    code_identity=source_identity(),
    software=software_versions(),
    weight_bytes=(ROOT / "examples/weights.csv").read_bytes(),
)
report_path = ROOT / "build" / f"notebook-{result['run_id']}.json"
publish(report_path, seal_report(result))
view = load_report(report_path)
display(view)

## Inspect exact tables

Signal metrics remain numeric; accounting amounts remain exact decimal strings. Charts use floating-point conversions only for display. Unavailable values remain null, not zero.

In [ ]:
display(view.table("signals"))
display(view.table("economics"))

## Individual figures

Available kinds: `correlations`, `coverage`, `returns`, `costs`. Each page contains at most 20 evaluation groups; use `page=1` for the next page when present. Each time-series model retains its instrument/contract identity.

In [ ]:
display(view.figure("correlations"))
display(view.figure("costs"))

## Export the same evidence

Creates private local HTML, SVG/PNG, CSV, source JSON and a verification manifest. No uploads or network calls. Do not commit real reports or notebook outputs.

In [ ]:
from red_five.rendering import verify_bundle

bundle = ROOT / "build" / f"notebook-bundle-{result['run_id']}"
html_path = view.export(bundle)
assert verify_bundle(bundle)["run_id"] == result["run_id"]
print(html_path)